In [ ]:
!pip install -q sentence-transformers pandas tqdm openpyxl

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from google.colab import files
from sentence_transformers import SentenceTransformer, util

In [ ]:
uploaded = files.upload()

file_names = list(uploaded.keys())
print(file_names)

Saving guide_master_skills.csv to guide_master_skills.csv
Saving video_skills_unified.csv to video_skills_unified.csv
['guide_master_skills.csv', 'video_skills_unified.csv']


In [ ]:
guide_file = [f for f in file_names if "guide" in f.lower()][0]
video_file = [f for f in file_names if "video" in f.lower()][0]

guide_df = pd.read_csv(guide_file)
video_df = pd.read_csv(video_file)

print("Guide columns:", guide_df.columns.tolist())
print("Video columns:", video_df.columns.tolist())

Guide columns: ['unit_number', 'lesson_number', 'lesson_title', 'skill_number', 'official_skill', 'source_type', 'notes']
Video columns: ['unit_number', 'lesson_number', 'official_lesson_title', 'original_lesson_title', 'normalized_lesson_title', 'lesson_match_score', 'lesson_match_method', 'lesson_needs_review', 'skill', 'skill_norm']


In [ ]:
GUIDE_LESSON_COL = "lesson_title"
GUIDE_SKILL_COL  = "official_skill"

VIDEO_LESSON_COL = "official_lesson_title"
VIDEO_SKILL_COL  = "skill"

In [ ]:
VIDEO_EXTRA_COLS = ["concept", "learning_objective"]

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).strip()
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)  # حذف التشكيل
    text = text.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")
    text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789"))
    text = re.sub(r"[(){}\[\]،,:؛.!؟\-_/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
guide_df["lesson_norm"] = guide_df[GUIDE_LESSON_COL].apply(clean_text)
guide_df["skill_norm"] = guide_df[GUIDE_SKILL_COL].apply(clean_text)

video_df["lesson_norm"] = video_df[VIDEO_LESSON_COL].apply(clean_text)
video_df["skill_norm"] = video_df[VIDEO_SKILL_COL].apply(clean_text)

In [ ]:
def build_video_text(row):
    parts = [row.get("skill_norm", "")]

    for col in VIDEO_EXTRA_COLS:
        if col in video_df.columns:
            parts.append(clean_text(row[col]))

    return " ".join([p for p in parts if p])

video_df["video_text"] = video_df.apply(build_video_text, axis=1)

In [ ]:
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def classify_match(score, guide_skill, video_skill):
    guide_skill = clean_text(guide_skill)
    video_skill = clean_text(video_skill)

    lexical_match = (
        guide_skill in video_skill or
        video_skill in guide_skill
    )

    if lexical_match and score >= 0.60:
        return "Covered by General Skill"

    if score >= 0.85:
        return "Exact/Semantic Match"

    elif score >= 0.70:
        return "Semantic Match"

    elif score >= 0.55:
        return "Partial Match"

    else:
        return "Missing"

In [ ]:
results = []

lessons = sorted(guide_df["lesson_norm"].dropna().unique())

for lesson in tqdm(lessons):

    guide_skills = (
        guide_df[guide_df["lesson_norm"] == lesson]["skill_norm"]
        .dropna()
        .unique()
        .tolist()
    )

    video_skills = (
        video_df[video_df["lesson_norm"] == lesson]["video_text"]
        .dropna()
        .unique()
        .tolist()
    )

    original_lesson_name = guide_df[
        guide_df["lesson_norm"] == lesson
    ][GUIDE_LESSON_COL].iloc[0]

    if len(video_skills) == 0:
        for guide_skill in guide_skills:
            results.append({
                "lesson_title": original_lesson_name,
                "guide_skill": guide_skill,
                "best_video_skill": "",
                "similarity_score": 0.0,
                "match_status": "Missing",
                "matched": False
            })
        continue

    guide_embeddings = model.encode(
        guide_skills,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    video_embeddings = model.encode(
        video_skills,
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    sim_matrix = util.cos_sim(guide_embeddings, video_embeddings)

    for i, guide_skill in enumerate(guide_skills):

        scores = sim_matrix[i].cpu().numpy()
        best_idx = int(np.argmax(scores))
        best_score = float(scores[best_idx])
        best_video_skill = video_skills[best_idx]

        match_status = classify_match(
            best_score,
            guide_skill,
            best_video_skill
        )

        matched = match_status in [
            "Exact/Semantic Match",
            "Semantic Match",
            "Covered by General Skill"
        ]

        results.append({
            "lesson_title": original_lesson_name,
            "guide_skill": guide_skill,
            "best_video_skill": best_video_skill,
            "similarity_score": round(best_score, 3),
            "match_status": match_status,
            "matched": matched
        })

100%|██████████| 41/41 [00:59<00:00,  1.46s/it]


In [ ]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    "skill_alignment_detailed_results.csv",
    index=False,
    encoding="utf-8-sig"
)

results_df.head()

,lesson_title,guide_skill,best_video_skill,similarity_score,match_status,matched
0,الأعداد ضمن 9999,قراءه الاعداد ضمن 9999 قراءه صحيحه,تمثيل الاعداد ضمن 9999 علي المعداد,0.820,Semantic Match,True
1,الأعداد ضمن 9999,كتابه الاعداد ضمن 9999 بالرموز,كتابه الاعداد ضمن 9999 بالرموز من الكلمات الصي...,0.865,Covered by General Skill,True
2,الأعداد ضمن 9999,كتابه الاعداد ضمن 9999 بالكلمات,كتابه الاعداد ضمن 9999 بالرموز من الكلمات الصي...,0.922,Exact/Semantic Match,True
3,الأعداد ضمن 9999,تمثيل الاعداد ضمن 9999 علي المعداد,تمثيل الاعداد ضمن 9999 علي المعداد,1.000,Covered by General Skill,True
4,الأعداد ضمن 9999,تمثيل الاعداد ضمن 9999 بلوحه المنازل,تمثيل الاعداد ضمن 9999 علي لوحه المنازل,0.941,Exact/Semantic Match,True


In [ ]:
total_guide_skills = len(results_df)
matched_guide_skills = results_df["matched"].sum()
missing_guide_skills = total_guide_skills - matched_guide_skills

coverage = matched_guide_skills / total_guide_skills if total_guide_skills > 0 else 0

print("Total Guide Skills:", total_guide_skills)
print("Matched Guide Skills:", matched_guide_skills)
print("Missing Guide Skills:", missing_guide_skills)
print("Coverage / Recall:", round(coverage, 3))

Total Guide Skills: 210
Matched Guide Skills: 148
Missing Guide Skills: 62
Coverage / Recall: 0.705


In [ ]:
video_matched = results_df[
    results_df["matched"] == True
]["best_video_skill"].nunique()

total_video_skills = video_df["video_text"].dropna().nunique()

precision = video_matched / total_video_skills if total_video_skills > 0 else 0

print("Total Video Skills:", total_video_skills)
print("Matched Video Skills:", video_matched)
print("Precision:", round(precision, 3))

Total Video Skills: 584
Matched Video Skills: 110
Precision: 0.188


In [ ]:
recall = coverage

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0 else 0
)

print("Precision:", round(precision, 3))
print("Recall / Coverage:", round(recall, 3))
print("F1-score:", round(f1, 3))

Precision: 0.188
Recall / Coverage: 0.705
F1-score: 0.297


In [ ]:
lesson_scores = results_df.groupby("lesson_title").agg(
    total_guide_skills=("guide_skill", "count"),
    matched_guide_skills=("matched", "sum"),
    average_similarity=("similarity_score", "mean")
).reset_index()

lesson_scores["coverage_score"] = (
    lesson_scores["matched_guide_skills"] /
    lesson_scores["total_guide_skills"]
).round(3)

lesson_scores["average_similarity"] = lesson_scores["average_similarity"].round(3)

lesson_scores.to_csv(
    "lesson_level_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

lesson_scores.head()

,lesson_title,total_guide_skills,matched_guide_skills,average_similarity,coverage_score
0,الأعداد ضمن 9999,6,6,0.889,1.000
1,الأعداد ضمن 99999,5,3,0.634,0.600
2,البيانات وتمثيلها بالصور,5,0,0.000,0.000
3,التقريب,11,9,0.858,0.818
4,الزاوية وأنواعها,5,0,0.348,0.000


In [ ]:
status_summary = results_df["match_status"].value_counts().reset_index()
status_summary.columns = ["match_status", "count"]

status_summary["percentage"] = (
    status_summary["count"] / len(results_df)
).round(3)

status_summary.to_csv(
    "match_status_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

status_summary

,match_status,count,percentage
0,Semantic Match,89,0.424
1,Exact/Semantic Match,41,0.195
2,Partial Match,41,0.195
3,Missing,21,0.100
4,Covered by General Skill,18,0.086


In [ ]:
files.download("skill_alignment_detailed_results.csv")
files.download("lesson_level_scores.csv")
files.download("match_status_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>